# Multiple Linear Regression From Scratch

## 1. Project Overview

The objective of this notebook is to understand and implement a **Multiple Linear Regression model from scratch**, using Python and NumPy, without relying on machine learning libraries such as Scikit-learn.

Building on the concepts learned in Simple Linear Regression, this project introduces the use of multiple independent variables to predict a single target variable.

We will implement the mathematical foundations of the algorithm, understand how weights and bias are calculated, and explore two optimization approaches:

- Ordinary Least Squares (OLS).
- Gradient Descent (GD).

## 2. What Is Multiple Linear Regression?

Multiple Linear Regression is a supervised machine learning algorithm used to model the relationship between **multiple independent variables (features) and one dependent variable (target)**.

Unlike Simple Linear Regression, which uses only one feature, Multiple Linear Regression considers several features to make predictions.

The mathematical representation is:

\[
$\hat{y} = w_1x_1 + w_2x_2 + \cdots + w_nx_n + b$
\]

Where:

- \($\hat{y}$\): Predicted target value.
- \($x_1, x_2, \ldots, x_n$\): Independent variables (features).
- \($w_1, w_2, \ldots, w_n$\): Weights (coefficients) assigned to each feature.
- \($b$\): Bias (intercept).
- \($n$\): Number of independent variables.

Each weight represents the contribution of its corresponding feature to the prediction, assuming the other features remain constant.

## 3. Mathematical Foundations

### 3.1. Cost Function

The objective of the model is to find the optimal weights and bias that minimize the difference between actual and predicted values.

We will use Mean Squared Error (MSE):

\[
$J(W,b)$ = $\frac{1}{m}\sum_{i=1}^{m}(\hat{y}^{(i)}-y^{(i)})^2$
\]

Where:

- \$(m\)$: Number of training examples.
- \$(y^{(i)}\)$: Actual target value.
- \$(\hat{y}^{(i)}\)$: Predicted target value.

### 3.2. Ordinary Least Squares (OLS)

OLS finds the optimal parameters by solving the minimization problem directly.

Using matrix notation:

\[
$\hat{W} = (X^TX)^{-1}X^Ty$
\]

This expression assumes that \$(X^TX\)$ is invertible. In practice, we will use `np.linalg.lstsq()` or the pseudo-inverse when necessary to handle singular or poorly conditioned matrices.

The bias can be included in the design matrix by adding a column of ones to the feature matrix.

### 3.3. Gradient Descent (GD)

Gradient Descent is an iterative optimization algorithm that updates the weights and bias to minimize the cost function.

**Weight gradients:**

\[
$\frac{\partial J}{\partial W}$ =
$\frac{2}{m}X^T(\hat{y}-y)$
\]

**Bias gradient:**

\[
$\frac{\partial J}{\partial b} =$
$\frac{2}{m}\sum_{i=1}^{m}(\hat{y}^{(i)}-y^{(i)})$
\]

**Parameter updates:**

\[
$W := W-\alpha\frac{\partial J}{\partial W}$
\]

\[
$b := b-\alpha\frac{\partial J}{\partial b}$
\]

Where \$(\alpha\)$ represents the learning rate.

Unlike OLS, Gradient Descent starts with initial parameter values and repeatedly updates them until convergence or until the maximum number of iterations is reached.

## 4. Implementation Roadmap

We will build the model through the following steps:

1. **Import Libraries:** Import NumPy, Pandas, and Matplotlib for numerical operations, data manipulation, and visualization.
2. **Create or Load Dataset:** Prepare a dataset containing multiple independent variables and one target variable.
3. **Exploratory Data Analysis (EDA):**
   - Inspect dataset dimensions and data types.
   - Identify missing values and duplicates.
   - Explore relationships between features and the target.
   - Visualize feature distributions and correlations.
4. **Data Preparation:**
   - Separate independent variables (X) from the target (y).
   - Split the dataset into training and testing sets.
   - Apply feature scaling when appropriate, especially for Gradient Descent.
5. **Implement Multiple Linear Regression Using OLS:**
   - Represent the dataset using matrix notation.
   - Calculate the optimal weights and bias.
   - Generate predictions.
6. **Implement Multiple Linear Regression Using Gradient Descent:**
   - Initialize weights and bias.
   - Calculate predictions and the cost function.
   - Calculate gradients.
   - Update weights and bias iteratively.
   - Track the cost across iterations.
7. **Evaluate the Models:** Use the following regression metrics:
   - Mean Absolute Error (MAE).
   - Mean Squared Error (MSE).
   - Root Mean Squared Error (RMSE).
   - R-squared (\$R^2$\).
8. **Compare Both Approaches:**
   - Compare learned weights and bias.
   - Compare predictions.
   - Compare model performance.
   - Analyze convergence and computational differences.
9. **Visualize Results:**
   - Plot actual versus predicted values.
   - Visualize the Gradient Descent cost history.
   - Analyze residuals.

## 5. Expected Outcomes

By the end of this notebook, we should be able to:

- Understand the mathematical foundations of Multiple Linear Regression.
- Understand the transition from a single feature to multiple features.
- Work with vectors and matrices to represent a regression problem.
- Implement OLS using NumPy.
- Implement Gradient Descent and understand how parameters are updated.
- Understand the role of feature scaling in Gradient Descent.
- Evaluate and compare regression models using appropriate metrics.
- Interpret the contribution of individual features through their learned coefficients.

---

**Important:** The goal of this project is to understand how Multiple Linear Regression works internally, rather than simply using an existing implementation. All model calculations and optimization steps will be implemented using NumPy.

In [3]:
import pandas as pd
import numpy as np
import math
from pathlib import Path

cwd = Path.cwd()
root = cwd.parent
data_dir = root / "Datasets" / "Multiple-linear-regression"

df = pd.read_csv(data_dir / "multiple_linear_regression_dataset.csv")

print(df.info())
print(df.describe())
print(df.shape)

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   age         20 non-null     int64
 1   experience  20 non-null     int64
 2   income      20 non-null     int64
dtypes: int64(3)
memory usage: 612.0 bytes
None
             age  experience        income
count  20.000000   20.000000     20.000000
mean   39.650000    6.200000  40735.500000
std    10.027725    4.124382   8439.797625
min    23.000000    1.000000  27840.000000
25%    31.500000    3.750000  35452.500000
50%    40.000000    5.000000  40190.000000
75%    47.000000    9.000000  45390.000000
max    58.000000   17.000000  63600.000000
(20, 3)


In [16]:
y = df["income"]
X = df.drop(columns=["income"])

def train_test_split(X,y,test_size):
    X_train = X.sample(frac=(1-test_size),random_state=42)
    y_train = y.sample(frac=(1-test_size),random_state=42)
    X_test = X.drop(X_train.index)
    y_test = y.drop(y_train.index)

    return X_train,X_test,y_train,y_test

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2
)

def scale(x):
    return (x-x.mean())/x.std()

def fit(X,y):
    X_scaled = scale(X)

    W = []

    for col in X_scaled.columns:
        w = sum((X_scaled[col] - X_scaled[col].mean()) * (y - y.mean())) / sum((X_scaled[col] - X_scaled[col].mean())**2)
        W.append(w)

    b = y.mean() - np.dot(X_scaled.mean(),W)

    return W , b

W , b = fit(X_train,y_train)

def predict(X):
    X_scaled = scale(X)

    return np.dot(X_scaled,W) + b


y_predict = predict(X_test)

RSS = sum((y_predict - y_test)**2)
RSE = np.sqrt(RSS/(len(y_test)-2))
TSS = sum((y_test - y_test.mean())**2)
R2 = 1 - (RSS/TSS)

print(f"RSS : {RSS}")
print(f"RSE : {RSE}")
print(f"TSS : {TSS}")
print(f"R2 : {R2}")

RSS : 81665396.32271092
RSE : 6390.046804316496
TSS : 281999475.0
R2 : 0.7104058568807232
